# Plot macrophage states and spatial expression 

**Goal.** Plot where the scANVI-transferred macrophage states relative to the EMT-like and
stem-like tumour states in the Visium HD data and and what they express.

**Input.** `adata_sp_final_relabelled_with_macs.h5ad`: raw counts in `X`, final labels in
`cell_label_final`

**Panels.**
1. Macrophage state composition per section (ER / SR / LR1 / LR2, pre / post).
2. All cell labels in space, one map per slide.
3. Key macrophage and tumour states on H&E, whole slide plus a zoom.
4. Macrophage marker dotplot (lipid, lysosomal, iron, scavenger, matrix programmes).
5. CD47 on tumour cells × SIRPA on macrophages (the "don't eat me" axis).

In [ ]:
# !pip install --quiet scanpy squidpy
# %cd <project>/spatial_visiumHD

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import squidpy as sq

## Config

In [ ]:
input_path = "adata_sp_final_relabelled_with_macs.h5ad"

figures_output_dir = "figures/macrophage"
tables_output_dir = "tables/macrophage"

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)

## 1. Load, colours and log-normalised layer

Raw counts stay in `X`; a `lognorm` layer (counts per 10k, log1p) is added for the expression panels.

In [ ]:
adata_sp = sc.read_h5ad(input_path)
if sample_col not in adata_sp.obs and "sample" in adata_sp.obs:
    sample_col = "sample"

vals = adata_sp.X[:1000].data if sp.issparse(adata_sp.X) else np.asarray(adata_sp.X[:1000]).ravel()
assert np.allclose(vals, np.round(vals)), "X must contain raw counts"


# --- columns ---
label_col = "cell_label_final"
slide_col = "slide"
sample_col = "sample_spatial"           # section, e.g. ER_pre ("sample" if built with notebook 1)
sample_order = ["ER_pre", "ER_post", "SR_pre", "SR_post",
                "LR1_pre", "LR1_post", "LR2_pre", "LR2_post"]   # response group = text before "_"


# --- colours: notebook-3 palette is kept in uns; these override it for the figure only ---
figure_color_overrides = {
    "4_Inflammatory EMT": "#fc1775",    # notebook 3: #fccde5
    "Macrophage CD36high": "#000000",   # #9c9ede
    "Macrophage INHBAhigh": "#f99506",  # #d33f6a
    "Macrophage STAB1high": "#58ff00",  # #99103d
}

adata_sp.obs[label_col] = adata_sp.obs[label_col].astype("category")
cats = list(adata_sp.obs[label_col].cat.categories)
stored = dict(zip(cats, adata_sp.uns.get(f"{label_col}_colors", [])))
palette = {**stored, **figure_color_overrides}
missing = [c for c in cats if c not in palette]
assert not missing, f"labels without a colour: {missing}"
adata_sp.uns[f"{label_col}_colors"] = [palette[c] for c in cats]

adata_sp.layers["lognorm"] = adata_sp.X.copy()
_tmp = sc.AnnData(adata_sp.layers["lognorm"])
sc.pp.normalize_total(_tmp, target_sum=1e4)
sc.pp.log1p(_tmp)
adata_sp.layers["lognorm"] = _tmp.X
del _tmp

pd.crosstab(adata_sp.obs[label_col], adata_sp.obs[slide_col], margins=True)

## 2. Macrophage state composition per section

Percentage of each macrophage state among all state-assigned macrophages in the section;
the number above each bar is the macrophage count (`Uncertain` excluded).

In [ ]:
# --- states ---
mac_states = ["Macrophage CD36high", "Macrophage STAB1high", "Macrophage INHBAhigh"]
overlay_labels = ["Macrophage CD36high", "Macrophage STAB1high",
                  "3_Devp_stem-like", "5_Neurod_EMT-like", "4_Inflammatory EMT"]

obs_mac = adata_sp.obs[adata_sp.obs[label_col].isin(mac_states)]
counts = (pd.crosstab(obs_mac[sample_col].astype(str), obs_mac[label_col].astype(str))
          .reindex(index=[s for s in sample_order if s in set(obs_mac[sample_col].astype(str))],
                   columns=mac_states, fill_value=0))
props = counts.div(counts.sum(axis=1), axis=0) * 100
pd.concat({"n": counts, "pct": props.round(1)}, axis=1).to_csv(f"{tables_output_dir}/macs_state_composition_by_sample.csv")

groups = props.index.str.split("_").str[0]
fig, ax = plt.subplots(figsize=(max(4, len(props) * 0.9), 4))
props.plot(kind="bar", stacked=True, color=[palette[s] for s in mac_states], width=0.75,
           edgecolor="none", ax=ax)
for i, n in enumerate(counts.sum(axis=1)):
    ax.text(i, 101, str(n), ha="center", va="bottom", fontsize=8)
for g in pd.unique(groups):                                  # group label + separator per response group
    idx = np.where(groups == g)[0]
    ax.text(idx.mean(), 109, g, ha="center", va="bottom", fontsize=12)
    if idx[0] > 0:
        ax.axvline(idx[0] - 0.5, color="grey", lw=1, ls="--", alpha=0.8)
ax.set_ylim(0, 115)
ax.set_xlabel("")
ax.set_ylabel("Proportion of macrophage states (%)", fontsize=12)
ax.tick_params(axis="x", labelrotation=90, labelsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Macrophage state", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
plt.savefig(f"{figures_output_dir}/macs_state_composition_by_sample.pdf", bbox_inches="tight")
plt.show()
counts

## 3. All cell labels in space (one map per slide)

In [ ]:
# --- spatial plots ---
point_size, img_alpha = 35, 0.6
zoom_boxes = {"slide2": (1000, 5500, 500, 2800)}   # x0, x1, y0 (top), y1 (bottom) in image pixels

slides = sorted(adata_sp.obs[slide_col].unique())
for slide in slides:
    fig, ax = plt.subplots(figsize=(8, 8))
    sq.pl.spatial_scatter(adata_sp[adata_sp.obs[slide_col] == slide], library_id=slide, img=False,
                          color=label_col, size=point_size, ax=ax)
    ax.set_title(slide)
    plt.savefig(f"{figures_output_dir}/all_labels_{slide}.pdf", bbox_inches="tight")
    plt.show()

## 4. Key macrophage and tumour states on H&E

Whole slide per slide, then a zoom for slides listed in `zoom_boxes` (image-pixel coordinates,
y increasing downwards as in the H&E image).

In [ ]:
def overlay(slide, zoom=None):
    sub = adata_sp[(adata_sp.obs[slide_col] == slide) & adata_sp.obs[label_col].isin(overlay_labels)].copy()
    sub.obs[label_col] = sub.obs[label_col].cat.remove_unused_categories()
    sub.uns[f"{label_col}_colors"] = [palette[c] for c in sub.obs[label_col].cat.categories]
    fig, ax = plt.subplots(figsize=(10, 10))
    sq.pl.spatial_scatter(sub, library_id=slide, img=True, img_alpha=img_alpha, color=label_col,
                          size=point_size if zoom is None else 20, linewidths=0 if zoom is None else 0.2,
                          edgecolor="black", legend_loc="right margin" if zoom is None else None, ax=ax)
    tag = "BIG"
    if zoom is not None:
        x0, x1, y0, y1 = zoom
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)            # top of the image = small y
        tag = "ZOOM"
    plt.savefig(f"{figures_output_dir}/key_celltypes_overlay_{tag}_{slide}.pdf", bbox_inches="tight", dpi=150)
    plt.show()


for slide in slides:
    overlay(slide)
    if slide in zoom_boxes:
        overlay(slide, zoom_boxes[slide])

## 5. Macrophage marker dotplot

Log-normalised expression, scaled 0–1 per gene across macrophage states.

In [ ]:
# --- marker dotplot (log-normalised expression, scaled per gene) ---
mac_markers = {
    "Lipid": ["APOE", "LIPA", "GPNMB", "SPP1", "CD36"],
    "Lysosomal": ["CTSB", "CTSD", "CTSL", "CD68"],
    "Iron": ["FTL", "FTH1"],
    "Scavenger": ["C1QC", "STAB1", "SELENOP", "F13A1"],
    "Matrix remodelling": ["COL1A1", "COL1A2", "COL3A1", "SPARC", "TIMP1"],
}

ad_mac = adata_sp[adata_sp.obs[label_col].isin(mac_states)].copy()
ad_mac.obs[label_col] = pd.Categorical(ad_mac.obs[label_col], categories=mac_states)
markers = {k: [g for g in v if g in ad_mac.var_names] for k, v in mac_markers.items()}
absent = sorted(set(sum(mac_markers.values(), [])) - set(sum(markers.values(), [])))
if absent:
    print("markers not in panel:", absent)

dp = sc.pl.dotplot(ad_mac, var_names=markers, groupby=label_col, layer="lognorm", cmap="Reds",
                   standard_scale="var", swap_axes=True, figsize=(3, 5), return_fig=True)
dp.make_figure()
for ax in dp.get_axes().values():
    for t in ax.get_yticklabels():
        t.set_fontstyle("italic")
dp.get_axes()["mainplot_ax"].set_xlabel("")
dp.fig.savefig(f"{figures_output_dir}/top_markers_macrophage_dotplot.pdf", bbox_inches="tight")
plt.show()

## 6. Ligand on tumour cells × receptor on macrophages

Grey: all selected tumour cells. Coloured: tumour cells expressing the ligand and macrophages
expressing the receptor (log-normalised, > 0). Colour scales are shared across slides
(`lr_vmin` to the `lr_vmax_percentile` of expressing cells), so slides are comparable.

In [ ]:
# --- ligand (tumour) x receptor (macrophage) map ---
lr_pairs = [("CD47", "SIRPA")]
lr_tumour_states = ["3_Devp_stem-like", "4_Inflammatory EMT", "5_Neurod_EMT-like"]
lr_mac_states = ["Macrophage CD36high", "Macrophage STAB1high"]
lr_cmaps = ("cool_r", "summer")
lr_vmin = 2.0                           # log-normalised floor of the colour scale (not a filter)
lr_vmax_percentile = 99                 # shared across slides

def expr(a, gene):
    x = a[:, gene].layers["lognorm"]
    return np.asarray(x.toarray() if sp.issparse(x) else x).ravel()

def plot_lr(ligand, receptor):
    tum = adata_sp[adata_sp.obs[label_col].isin(lr_tumour_states)]
    mac = adata_sp[adata_sp.obs[label_col].isin(lr_mac_states)]
    e_t, e_m = expr(tum, ligand), expr(mac, receptor)
    vmax_t = np.percentile(e_t[e_t > 0], lr_vmax_percentile)
    vmax_m = np.percentile(e_m[e_m > 0], lr_vmax_percentile)
    print(f"{ligand}: {lr_vmin:.2f}-{vmax_t:.2f} | {receptor}: {lr_vmin:.2f}-{vmax_m:.2f}")

    for slide in slides:
        st = (tum.obs[slide_col] == slide).to_numpy()
        sm = (mac.obs[slide_col] == slide).to_numpy()
        xy_t, xy_m = tum.obsm["spatial"], mac.obsm["spatial"]
        on_t, on_m = st & (e_t > 0), sm & (e_m > 0)

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(xy_t[st, 0], xy_t[st, 1], color="lightgrey", s=15, alpha=0.25, lw=0, zorder=1)
        h_t = ax.scatter(xy_t[on_t, 0], xy_t[on_t, 1], c=e_t[on_t], cmap=lr_cmaps[0], vmin=lr_vmin,
                         vmax=vmax_t, s=22, lw=0, alpha=0.85, zorder=2)
        h_m = ax.scatter(xy_m[on_m, 0], xy_m[on_m, 1], c=e_m[on_m], cmap=lr_cmaps[1], vmin=lr_vmin,
                         vmax=vmax_m, s=30, lw=0.4, edgecolors="black", zorder=3)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.axis("off")
        ax.set_title(f"{slide}: {on_t.sum()} {ligand}+ tumour, {on_m.sum()} {receptor}+ macrophages", fontsize=9)
        plt.colorbar(h_t, ax=ax, fraction=0.025, pad=0.02).set_label(f"{ligand} (tumour, log-norm)")
        plt.colorbar(h_m, ax=ax, fraction=0.025, pad=0.10).set_label(f"{receptor} (macrophage, log-norm)")
        plt.savefig(f"{figures_output_dir}/{ligand}_{receptor}_{slide}.pdf", bbox_inches="tight")
        plt.show()


for ligand, receptor in lr_pairs:
    plot_lr(ligand, receptor)